# Embedding

## Reading and Preprocessing data

In [81]:
MED_DATA_PATH = "./TP_ISD2020/QUAERO_FrenchMed/QUAERO_FrenchMed_traindev.ospl"
PRESS_DATA_PATH = "./TP_ISD2020/QUAERO_FrenchPress/QUAERO_FrenchPress_traindev.ospl"

In [82]:
# Load the data file 
def load_data(Path):
    with open(Path) as f:
        data = f.readlines()
    return data

# Preprocess the text by tokenizing it into the input format of the models
def preprocess_data(data):
    tokenized_data = [line.lower().strip().split() for line in data]
    return tokenized_data

In [83]:
# Load and preprocess the data

med_data = load_data(MED_DATA_PATH)
press_data = load_data(PRESS_DATA_PATH)

tokenized_med_data = preprocess_data(med_data)
tokenized_press_data = preprocess_data(press_data)

In [84]:
# Preview of the results for medical data
tokenized_med_data[0:5]  # Display the first 5 tokenized sentences from the medical data

[['emea', '/', 'h', '/', 'c', '/', '551'],
 ['prialt'],
 ['qu', '’', 'est', 'ce', 'que', 'prialt', '?'],
 ['prialt',
  'est',
  'une',
  'solution',
  'pour',
  'perfusion',
  'contenant',
  'le',
  'principe',
  'actif',
  'ziconotide',
  ',',
  'à',
  'des',
  'concentrations',
  'de',
  '100',
  'ou',
  '25',
  'microgrammes',
  'par',
  'millilitre',
  '.'],
 ['dans', 'quel', 'cas', 'prialt', 'est', '-', 'il', 'utilisé', '?']]

In [85]:
# Preview of the results for press data
tokenized_press_data[0:5]  # Display the first 5 tokenized sentences from the medical data

[['patricia',
  'martin',
  ',',
  'que',
  'voici',
  ',',
  'que',
  'voilà',
  '!',
  'oh',
  ',',
  'bonjour',
  'nicolas',
  'stoufflet',
  '.'],
 ['france', 'inter', ',', '7', 'heures', '.'],
 ['le', 'journal', ',', 'simon', 'tivolle', '.'],
 ['bonjour', '!', 'lundi', '7', 'décembre', '.'],
 ['deux',
  'incendies',
  'cette',
  'nuit',
  'en',
  'région',
  'parisienne',
  ',',
  'dans',
  'une',
  'maison',
  'de',
  'retraite',
  'de',
  'livry-gargan',
  'en',
  'seine-saint-denis',
  ',',
  '7',
  'personnes',
  'ont',
  'péri',
  'dans',
  'les',
  'flammes',
  '.']]

## Word Embeddings Training 

### Word2Vec Models

In [86]:
from gensim.models import Word2Vec

# Helper function to train Word2Vec models with specified parameters
def w2v(sentences, sg, vector_size=100, window=5):

	model = Word2Vec(
		sentences=sentences,
		vector_size=vector_size,
		window=window,
		min_count=1,
		sg=sg
	)
	return model

# CBOW model for medical data
w2v_cbow_med_model = w2v(tokenized_med_data, sg=0)

# Skipgram model for medical data
w2v_skipgram_med_model = w2v(tokenized_med_data, sg=1)

# CBOW model for press data
w2v_cbow_press_model = w2v(tokenized_press_data, sg=0)

# Skipgram model for press data
w2v_skipgram_press_model = w2v(tokenized_press_data, sg=1)


### Fast Text models

In [87]:
from gensim.models import FastText

# Helper function to create a FastText model with the specified parameters
def fasttext_cbow(sentences, vector_size=100, window=5, sg=0):
    model = FastText(
		sentences=sentences,
		vector_size=vector_size,
		window=window,
		min_count=1,
		sg=sg
	)
    return model

# CBOW model for medical data using FastText
ft_cbow_med_model = fasttext_cbow(tokenized_med_data)

# CBOW model for press data using FastText
ft_cbow_press_model = fasttext_cbow(tokenized_press_data)

### Saving the embeddings

In [ ]:
import os

os.makedirs("embeddings", exist_ok=True)

all_models = {
    "w2v_cbow_med": w2v_cbow_med_model,
    "w2v_skipgram_med": w2v_skipgram_med_model,
    "ft_cbow_med": ft_cbow_med_model,
    "w2v_cbow_press": w2v_cbow_press_model,
    "w2v_skipgram_press": w2v_skipgram_press_model,
    "ft_cbow_press": ft_cbow_press_model,
}

for name, model in all_models.items():
    model.wv.save_word2vec_format(f"embeddings/{name}.bin", binary=True)

### Sanity check

In [88]:
# Sanity check function to verify the models and their word vectors

med_models = [w2v_cbow_med_model, w2v_skipgram_med_model, ft_cbow_med_model]
press_models = [w2v_cbow_press_model, w2v_skipgram_press_model, ft_cbow_press_model]

def sanityquick_check(models, words):
	for model in models:
		print(f"Model: {model}")
		for word in words:
			if word in model.wv:
				print(f"Word: {word}, Vector: {model.wv[word][:5]}, Vector Length: {len(model.wv[word])}")
			else:
				print(f"Word: {word} not found in the model.")
		print("\n")
  

In [89]:
# Sanity check for medical models with a list of relevant words
sanityquick_check(med_models, ["cancer", "santé", "maladie", "traitement", "virus"])

Model: Word2Vec<vocab=8088, vector_size=100, alpha=0.025><vocab=8088, vector_size=100, alpha=0.025><vocab=8088, vector_size=100, alpha=0.025>
Word: cancer, Vector: [-0.09739568  0.16489673  0.07880066  0.07313921  0.02472525], Vector Length: 100
Word: santé, Vector: [-0.03569639  0.07105725  0.01845646  0.03263839  0.00167529], Vector Length: 100
Word: maladie, Vector: [-0.1913242   0.31492093  0.13351423  0.14066687  0.02473816], Vector Length: 100
Word: traitement, Vector: [-0.30052552  0.47212708  0.2057795   0.1958217   0.02095164], Vector Length: 100
Word: virus, Vector: [-0.08535806  0.14114018  0.05358052  0.04943257  0.00980386], Vector Length: 100


Model: Word2Vec<vocab=8088, vector_size=100, alpha=0.025><vocab=8088, vector_size=100, alpha=0.025><vocab=8088, vector_size=100, alpha=0.025>
Word: cancer, Vector: [-0.22159098  0.18845822  0.22057115  0.09775233 -0.01480197], Vector Length: 100
Word: santé, Vector: [-0.11044665  0.16943727  0.09885015  0.0631306  -0.06786036], Vec

In [90]:
# Sanity check for press models with a list of relevant words
sanityquick_check(press_models, ["politique", "économie", "sport", "culture", "technologie"])

Model: Word2Vec<vocab=8088, vector_size=100, alpha=0.025><vocab=8088, vector_size=100, alpha=0.025><vocab=38279, vector_size=100, alpha=0.025>
Word: politique, Vector: [-1.0893891  -0.9166306   0.6719969   1.153249    0.09792129], Vector Length: 100
Word: économie, Vector: [-0.5639193  -0.40267268 -0.12934978  0.05228897 -0.37280765], Vector Length: 100
Word: sport, Vector: [ 0.04563551  0.41637528  0.598889    0.5246265  -0.23693687], Vector Length: 100
Word: culture, Vector: [-0.57259643 -0.13919897  0.21246433 -0.05091333 -0.09356114], Vector Length: 100
Word: technologie, Vector: [-0.02303978  0.12644315  0.03365678  0.01348368  0.03006399], Vector Length: 100


Model: Word2Vec<vocab=8088, vector_size=100, alpha=0.025><vocab=8088, vector_size=100, alpha=0.025><vocab=38279, vector_size=100, alpha=0.025>
Word: politique, Vector: [-0.35136434 -0.1566942   0.11469537 -0.13071258 -0.15795815], Vector Length: 100
Word: économie, Vector: [-0.25104287  0.14439078  0.23344475  0.00207955  0

## Semantic Similarity

### Comparaison of the embeddings trained on the same corpus

In [95]:
cadidat_words = ['patient', 'traitement', 'maladie', 'solution', 'jaune']

# Function to get word embeddings for a list of words from multiple models

def get_word_embeddings(models, words):
	embeddings = {}
	for model in models:
		embeddings[model] = {}
		for word in words:
			if word in model.wv:
				embeddings[model][word] = model.wv[word]
			else:
				embeddings[model][word] = None
	return embeddings



In [96]:
# Get word embeddings for the candidate words from the medical models
word_embeddings_med = get_word_embeddings(med_models, cadidat_words)

In [ ]:
def compare_embeddings(embeddings):
	for model, words in embeddings.items():
		print(f"Model: {model}")
		word_embeddings = [embeddings[model][word] for word in cadidat_words]

In [97]:
# 10 Most similar words for each candidate word in the medical models

def get_most_similar_words(models, words, topn=10):
	model_results = []
 
	for model in models:
		similar_words = {}
		for word in words:
			if word in model.wv:
				similar_words[word] = model.wv.most_similar(word, topn=topn)
			else:
				similar_words[word] = None
		model_results.append(similar_words)
	return model_results

In [101]:
import pandas as pd
# CBOW model results for the candidate words in the medical models
med_model_results = get_most_similar_words(med_models, cadidat_words)

cbow_similar_words = pd.DataFrame(med_model_results[0])
cbow_similar_words

,patient,traitement,maladie,solution,jaune
0,"(le, 0.9992056488990784)","(le, 0.9996695518493652)","(du, 0.999712347984314)","(contient, 0.9992324113845825)","(), 0.9976211190223694)"
1,"(-, 0.9992024302482605)","(par, 0.9996500015258789)","(un, 0.9996981620788574)","((, 0.9991666078567505)","(8, 0.9976071715354919)"
2,"(dans, 0.9991958737373352)","(avec, 0.9996414184570312)","(la, 0.9996957778930664)","(2, 0.9991650581359863)","((, 0.9975560903549194)"
3,"(pour, 0.9991888403892517)","(des, 0.9996405839920044)","(de, 0.9996932744979858)","(1, 0.9991326332092285)","(-, 0.9975310564041138)"
4,"(et, 0.9991757273674011)","(dans, 0.9996331334114075)","(en, 0.9996833205223083)","(perfusion, 0.9991109371185303)","(2, 0.9975039958953857)"
5,"(avec, 0.9991679787635803)","(les, 0.9996240139007568)","(dans, 0.9996810555458069)","(flacon, 0.9991092681884766)","(entre, 0.9974608421325684)"
6,"(par, 0.9991647601127625)","(sur, 0.9996114373207092)","(à, 0.9996786713600159)","(), 0.9990977048873901)","(., 0.997459888458252)"
7,"(à, 0.9991600513458252)","(en, 0.9996069073677063)","(et, 0.9996746778488159)","(pour, 0.9990336298942566)","(pour, 0.9974390864372253)"
8,"(ou, 0.9991585612297058)","(un, 0.999606728553772)","(une, 0.999669075012207)","(,, 0.9989362359046936)","(1, 0.9974365830421448)"
9,"(que, 0.9991551637649536)","(une, 0.9996014833450317)","(des, 0.9996632933616638)","(0, 0.9989350438117981)","(:, 0.997419536113739)"


In [102]:
# Skipgram model results for the candidate words in the medical models
skipgram_similar_words = pd.DataFrame(med_model_results[1])
skipgram_similar_words

,patient,traitement,maladie,solution,jaune
0,"(qui, 0.9964388608932495)","(tysabri, 0.9687711000442505)","(résultats, 0.991736114025116)","(flacon, 0.9889858365058899)","(puis, 0.9980816841125488)"
1,"(symptômes, 0.9952467679977417)","(médicaments, 0.9646313190460205)","(association, 0.9916042685508728)","(100, 0.9792366027832031)","(23, 0.9978795647621155)"
2,"(associé, 0.9951844811439514)","(le, 0.9633438587188721)","(étude, 0.9900357127189636)","(contient, 0.9792014956474304)","(comprises, 0.9978027939796448)"
3,"(recommandé, 0.9949939250946045)","(risque, 0.9624570608139038)","(adulte, 0.9889194369316101)","(jour, 0.9786295294761658)","(urticaire, 0.9977021813392639)"
4,"(grossesse, 0.9947611689567566)","(association, 0.9605786800384521)","(prise, 0.9881195425987244)","(300, 0.9768068790435791)","(24, 0.9976677298545837)"
5,"(réactions, 0.9942825436592102)","(maladie, 0.9588059186935425)","(infection, 0.9880359172821045)","(fois, 0.976198673248291)","(thermoformées, 0.9976205825805664)"
6,"(avant, 0.9941914677619934)","(utilisation, 0.957599937915802)","(un, 0.9874960780143738)","(mg, 0.9748759865760803)","(dioxyde, 0.9975733757019043)"
7,"(nécessaire, 0.9935752749443054)","(pendant, 0.956500768661499)","(utilisation, 0.9873310923576355)","(150, 0.9732523560523987)","(80, 0.9975513219833374)"
8,"(arrêt, 0.9931573271751404)","(par, 0.9559392929077148)","(en, 0.9864648580551147)","(9, 0.9731884002685547)","(polysorbate, 0.9975135326385498)"
9,"(quel, 0.9927835464477539)","(prise, 0.9547099471092224)","(charge, 0.9861699342727661)","(6, 0.9731853604316711)","(équivalent, 0.9975079298019409)"


In [103]:
# FastText model results for the candidate words in the medical models
fasttext_similar_words = pd.DataFrame(med_model_results[2])
fasttext_similar_words

,patient,traitement,maladie,solution,jaune
0,"(patiente, 0.9999982714653015)","(taaitement, 0.9999983906745911)","(malade, 0.9999949336051941)","(dissolution, 0.9999974370002747)","(une, 0.9999914169311523)"
1,"(présentaient, 0.9999933838844299)","(allaitement, 0.9999977350234985)","(professionnelle, 0.9999929666519165)","(évolution, 0.9999967813491821)","(veine, 0.9999904036521912)"
2,"(aient, 0.9999933242797852)","(traitment, 0.9999971985816956)","(professionnel, 0.999991774559021)","(evolution, 0.9999961256980896)","(chine, 0.9999902844429016)"
3,"(conscients, 0.9999927282333374)","(évitement, 0.999997079372406)","(male, 0.999991774559021)","(dilution, 0.9999948143959045)","(routine, 0.9999902248382568)"
4,"(patients, 0.999992311000824)","(étroitement, 0.999997079372406)","(provoque, 0.9999915957450867)","(pollution, 0.9999939203262329)","(paraprotéine, 0.9999898672103882)"
5,"(avaient, 0.999992311000824)","(traitements, 0.9999968409538269)","(matière, 0.9999914169311523)","(exécution, 0.9999898672103882)","(lanoline, 0.9999898076057434)"
6,"(étaient, 0.9999921321868896)","(ajustement, 0.9999955892562866)","(professionnels, 0.999991238117218)","(élocution, 0.9999898672103882)","(proline, 0.9999896883964539)"
7,"(pansements, 0.9999919533729553)","(hautement, 0.9999952912330627)","(enceinte, 0.9999910593032837)","(diminution, 0.9999896287918091)","(lapine, 0.9999893307685852)"
8,"(prélèvements, 0.9999916553497314)","(jugement, 0.9999948740005493)","(tensionnelle, 0.999990701675415)","(substitution, 0.9999883770942688)","(humaine, 0.9999892711639404)"
9,"(recevaient, 0.9999916553497314)","(lentement, 0.9999947547912598)","(lésionnelle, 0.999990701675415)","(distribution, 0.9999881982803345)","(angine, 0.9999890327453613)"


In [104]:
press_model_results = get_most_similar_words(press_models, cadidat_words)
cbow_similar_words_press = pd.DataFrame(press_model_results[0])
cbow_similar_words_press

,patient,traitement,maladie,solution,jaune
0,"(effort, 0.8470407724380493)","(financement, 0.9373219609260559)","(population, 0.9251877069473267)","(difficulté, 0.8987979888916016)","(maillot, 0.9531394243240356)"
1,"(permettrait, 0.8427736163139343)","(coût, 0.9235988259315491)","(règle, 0.9167547225952148)","(forme, 0.8903883695602417)","(kafelnikov, 0.9433342218399048)"
2,"(annuler, 0.8377054333686829)","(calendrier, 0.9173473119735718)","(franco-américaine, 0.9128770232200623)","(génération, 0.8775463104248047)","(emmy, 0.942142128944397)"
3,"(accepter, 0.8328301310539246)","(programme, 0.9164369106292725)","(législation, 0.9069852232933044)","(discussion, 0.873959481716156)","(trophée, 0.9414337873458862)"
4,"(exercer, 0.8325236439704895)","(pacte, 0.9125379920005798)","(menace, 0.9059994220733643)","(censure, 0.873456597328186)","(dark, 0.9404502511024475)"
5,"(jeter, 0.8247288465499878)","(trafic, 0.9121106863021851)","(douleur, 0.9055700302124023)","(législation, 0.8734182119369507)","(von, 0.9402932524681091)"
6,"(potentiel, 0.8209570646286011)","(trésor, 0.9107739329338074)","(méthode, 0.905428946018219)","(leçon, 0.8726111054420471)","(oscar, 0.9400445222854614)"
7,"(fouetter, 0.8167711496353149)","(effectif, 0.9080882668495178)","(voie, 0.904517412185669)","(tradition, 0.8723724484443665)","(doh, 0.939912736415863)"
8,"(airbus, 0.8165301084518433)","(logement, 0.9074488282203674)","(dictature, 0.9030342102050781)","(meilleure, 0.8716540932655334)","(conservateur, 0.936523973941803)"
9,"(diplôme, 0.8157470226287842)","(cancer, 0.9047626852989197)","(relation, 0.8975522518157959)","(menace, 0.8711037635803223)","(samuel, 0.9362665414810181)"


In [105]:
skipgram_similar_words_press = pd.DataFrame(press_model_results[1])
skipgram_similar_words_press                                    

,patient,traitement,maladie,solution,jaune
0,"(motivation, 0.9525114297866821)","(coût, 0.9128511548042297)","(garantie, 0.8427643775939941)","(pacifique, 0.8725006580352783)","(maillot, 0.9254534840583801)"
1,"(adaptation, 0.9471824765205383)","(mépris, 0.9081348180770874)","(lourde, 0.8261357545852661)","(règle, 0.8690851926803589)","(quadruple, 0.8875693678855896)"
2,"(ordinateur, 0.9470859169960022)","(trésor, 0.9060632586479187)","(liée, 0.8238162994384766)","(alternative, 0.8676834106445312)","(spadea, 0.8865947723388672)"
3,"(mobylette, 0.9440614581108093)","(potentiel, 0.9012589454650879)","(trace, 0.8231300115585327)","(catastrophe, 0.8665653467178345)","(bastad, 0.8791642189025879)"
4,"(rajouter, 0.9439194202423096)","(alimentaire, 0.8971482515335083)","(facture, 0.822058916091919)","(légitimité, 0.8662711977958679)","(emparé, 0.8777245283126831)"
5,"(trou, 0.9424411058425903)","(pacte, 0.8949559330940247)","(totale, 0.8183106780052185)","(manoeuvre, 0.8628650903701782)","(mark, 0.8754849433898926)"
6,"(coq, 0.94126296043396)","(calendrier, 0.8927455544471741)","(faiblesse, 0.8172577619552612)","(claire, 0.861354410648346)","(robredo, 0.8734102249145508)"
7,"(brevet, 0.9412104487419128)","(consensus, 0.8917154669761658)","(contamination, 0.8164552450180054)","(vraie, 0.8611292839050293)","(demi-finaliste, 0.8724046349525452)"
8,"(truc, 0.9411220550537109)","(collectif, 0.8908876776695251)","(compensation, 0.8142613768577576)","(référence, 0.8594746589660645)","(requin, 0.8712337017059326)"
9,"(provocation, 0.9406105279922485)","(financement, 0.8854302167892456)","(pauvreté, 0.8132601380348206)","(puissance, 0.8593962788581848)","(oscar, 0.869533896446228)"


In [106]:
fasttext_similar_words_press = pd.DataFrame(press_model_results[2])
fasttext_similar_words_press

,patient,traitement,maladie,solution,jaune
0,"(patientent, 0.9902471303939819)","(farouchement, 0.9852029085159302)","(malnutrie, 0.9647155404090881)","(révolution, 0.9922932386398315)","(abdoune, 0.9709694385528564)"
1,"(trient, 0.9878193736076355)","(concrètement, 0.9851083159446716)","(trilogie, 0.9594129920005798)","(résolution, 0.9922788739204407)","(saadoune, 0.9706295728683472)"
2,"(impatient, 0.9857651591300964)","(promptement, 0.98406583070755)","(phraséologie, 0.9563109278678894)","(caution, 0.9882532358169556)","(saâdoune, 0.9700713157653809)"
3,"(répandent, 0.9848642945289612)","(fondement, 0.9833773374557495)","(réfléchie, 0.9533631801605225)","(évolution, 0.9879996180534363)","(brune, 0.9692159295082092)"
4,"(prient, 0.9846689105033875)","(fermement, 0.983262300491333)","(machinerie, 0.9529799818992615)","(dilution, 0.9865683317184448)","(lune, 0.9675580859184265)"
5,"(détient, 0.9845499396324158)","(rapidement, 0.9825865030288696)","(ossétie, 0.9529203772544861)","(pollution, 0.9859647154808044)","(lagune, 0.9507418274879456)"
6,"(coefficient, 0.9845362305641174)","(vachement, 0.9820683598518372)","(psycho-oncologie, 0.9519962072372437)","(dissolution, 0.9848817586898804)","(laâyoune, 0.9472284913063049)"
7,"(abstient, 0.9833031296730042)","(doucement, 0.9815419316291809)","(lockerbie, 0.9516525864601135)","(potion, 0.980124294757843)","(dune, 0.9283336400985718)"
8,"(contient, 0.9831900596618652)","(sèchement, 0.9808204770088196)","(tromperie, 0.9503254890441895)","(réception, 0.9796035885810852)","(l'une, 0.9269054532051086)"
9,"(conjuguent, 0.9828367829322815)","(dévouement, 0.9804085493087769)","(théologie, 0.9503129124641418)","(démolition, 0.9786946773529053)","(aune, 0.9112864136695862)"
